# Evaluasi model lgbm-v1

Metrik dari `models/lgbm-v1.json`, hasil `python -m padamfirst.model.train`. Validasi 2023 dipakai untuk tuning Optuna dan kalibrasi isotonic, uji 2024 sampai 2025 tidak pernah dilihat model. Baseline adalah `p_spread` baseline-v1 dari `scoring.py`.

Jalankan ulang dari `pipeline/`:

```bash
uv run --group notebook jupyter nbconvert --to notebook --execute --inplace notebooks/model.ipynb
```

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Paket padamfirst tidak di-install, notebook dijalankan dari pipeline/notebooks/.
sys.path.insert(0, str(Path.cwd().parent))
from padamfirst.model.train import META_PATH

meta = json.loads(META_PATH.read_text())
print(meta["years"])
print("Parameter terbaik:", meta["params"], "iterasi", meta["best_iteration"])

## PR-AUC, ROC-AUC, dan recall 50 sel teratas per hari

In [ ]:
table = pd.concat(
    {
        name: pd.DataFrame({k: result[k] for k in ("model", "baseline")})
        for name, result in meta["metrics"].items()
    }
)
table["selisih"] = table["model"] - table["baseline"]
table.round(4)

In [ ]:
test = meta["metrics"]["test"]
beats = test["model"]["pr_auc"] > test["baseline"]["pr_auc"]
print(f"Uji: {test['rows']} baris, label 1 {test['label_rate']:.3f}")
print("Model mengalahkan baseline di PR-AUC uji:", beats)

## Reliability plot

Titik dekat garis diagonal berarti probabilitas terkalibrasi sesuai frekuensi api yang bertahan atau menyebar.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], color="grey", linestyle="--", label="sempurna")
for name, result in meta["metrics"].items():
    curve = result["reliability"]
    ax.plot(curve["predicted"], curve["observed"], marker="o", label=name)
ax.set_xlabel("Probabilitas prediksi")
ax.set_ylabel("Frekuensi label 1")
ax.legend()
plt.show()